**Packages needed to be imported**

In [1]:
import numpy as np
import pandas as pd
from tensorflow.keras import layers,models
import os
from skimage.io import imread
from skimage.transform import resize
from sklearn.model_selection import train_test_split
from PIL import Image,ImageOps
import tensorflow as tf

In [2]:
!pip install imageio[gdal]  # Ensure GDAL is installed for imageio

import numpy as np
import os
from skimage.transform import resize
from PIL import Image

# Define input directory and categories
input_dir = r'/content/drive/MyDrive/PetImages'
categories = ['Cat', 'Dog']
data = []
label = []

for category_idx, category in enumerate(categories):
    category_path = os.path.join(input_dir, category)

    # Check if the directory exists before processing
    if not os.path.exists(category_path):
        print(f"Warning: Directory {category_path} does not exist!")
        continue

    for file in os.listdir(category_path):
        img_path = os.path.join(category_path, file)

        # Process only image files
        if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
            try:
                # Open image using PIL to ensure compatibility
                with Image.open(img_path) as img:
                    img = img.convert('RGB')  # Ensure 3-channel RGB format

                # Convert image to numpy array and resize
                img = np.array(img)
                img = resize(img, (15, 15))  # Resize to 15x15

                data.append(img.flatten())
                label.append(category_idx)

            except (OSError, ValueError, AttributeError) as e:
                print(f"Skipping problematic file: {file} due to {type(e).__name__}: {e}")
        else:
            print(f"Skipping non-image file: {file}")

# Convert lists to NumPy arrays
data = np.array(data)
label = np.array(label)

print("Data processing complete. Shape of data:", data.shape)
print("Number of labels:", len(label))


Skipping non-image file: Thumbs.db
Skipping problematic file: 666.jpg due to UnidentifiedImageError: cannot identify image file '/content/drive/MyDrive/PetImages/Cat/666.jpg'
Skipping non-image file: Thumbs.db


/usr/local/lib/python3.11/dist-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Skipping problematic file: 11702.jpg due to UnidentifiedImageError: cannot identify image file '/content/drive/MyDrive/PetImages/Dog/11702.jpg'
Data processing complete. Shape of data: (25019, 675)
Number of labels: 25019


In [3]:
X_train,X_test,y_train,y_test=train_test_split(data,label,test_size=0.20,random_state=22)

In [4]:
data_aggumentation=tf.keras.Sequential([
    layers.RandomZoom(0.1),
    layers.RandomRotation(0.10),
    layers.RandomFlip()
])

In [7]:
X_train = X_train.reshape(-1, 15, 15, 3)  # Reshape to (num_samples, 15, 15, 3)
X_test = X_test.reshape(-1, 15, 15, 3)

model = models.Sequential([
    layers.Conv2D(64, (3, 3), activation='relu', input_shape=(32, 32, 3), padding="same"),
    layers.MaxPooling2D((2, 2), strides=(2, 2)),  # Keeps spatial size manageable

    layers.Conv2D(128, (3, 3), activation='relu', padding="same"),
    layers.MaxPooling2D((2, 2), strides=(2, 2)),  # Avoids reducing below 1x1

    layers.Conv2D(256, (3, 3), activation='relu', padding="same"),
    layers.MaxPooling2D((2, 2), strides=(2, 2)),  # Ensure it does not shrink too much

    layers.Conv2D(512, (3, 3), activation='relu', padding="same"),
    layers.GlobalAveragePooling2D(),  # Replaces MaxPooling to prevent 1x1 issue

    layers.Dense(256, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

model.summary()


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)                    │ (None, 32, 32, 64)          │           1,792 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_3 (MaxPooling2D)       │ (None, 16, 16, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_5 (Conv2D)                    │ (None, 16, 16, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_4 (MaxPooling2D)       │ (None, 8, 8, 128)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_6 (Conv2D)                    │ (None, 8, 8, 256)           │         295,168 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_5 (MaxPooling2D)       │ (None, 4, 4, 256)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_7 (Conv2D)                    │ (None, 4, 4, 512)           │       1,180,160 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d_1           │ (None, 512)                 │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 256)                 │         131,328 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 10)                  │           2,570 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,684,874 (6.43 MB)

 Trainable params: 1,684,874 (6.43 MB)

 Non-trainable params: 0 (0.00 B)

In [8]:
history=model.fit(X_train,y_train,epochs=50,validation_data=(X_test,y_test))

Epoch 1/50
626/626 ━━━━━━━━━━━━━━━━━━━━ 13s 11ms/step - accuracy: 0.5055 - loss: 0.7630 - val_accuracy: 0.6197 - val_loss: 0.6648
Epoch 2/50
626/626 ━━━━━━━━━━━━━━━━━━━━ 13s 5ms/step - accuracy: 0.6388 - loss: 0.6349 - val_accuracy: 0.6477 - val_loss: 0.6135
Epoch 3/50
626/626 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.7019 - loss: 0.5696 - val_accuracy: 0.7300 - val_loss: 0.5444
Epoch 4/50
626/626 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.7287 - loss: 0.5329 - val_accuracy: 0.7036 - val_loss: 0.5634
Epoch 5/50
626/626 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.7489 - loss: 0.5141 - val_accuracy: 0.7096 - val_loss: 0.5682
Epoch 6/50
626/626 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.7688 - loss: 0.4776 - val_accuracy: 0.7468 - val_loss: 0.5124
Epoch 7/50
626/626 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.7843 - loss: 0.4556 - val_accuracy: 0.7484 - val_loss: 0.5052
Epoch 8/50
626/626 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.7928 - loss: 0.4298 - val_accuracy:

In [10]:
model.save('cat_dog_classifier.keras')

In [13]:
def predict_image(image_path, model):
    from tensorflow.keras.preprocessing import image
    import numpy as np

    # Define IMG_SIZE here or before calling the function
    IMG_SIZE = (32, 32)  # Match the input shape of your model

    img = image.load_img(image_path, target_size=IMG_SIZE)
    img_array = image.img_to_array(img) / 255.0  # Normalize
    img_array = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_array)[0][0]
    class_label = 'Dog' if prediction > 0.5 else 'Cat'
    confidence = prediction if prediction > 0.5 else 1 - prediction

    return class_label, confidence

In [14]:
class_label, confidence = predict_image('/content/Dog_Breeds.jpg', model)
print(f'Predicted: {class_label} with confidence: {confidence:.2f}')

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 744ms/step
Predicted: Cat with confidence: 1.00
